<center><h1>Трансферное обучение</h1>

— использование данных полученных с одного обучающего набора на данные с другого набора

<h2>Загрузка и обучение модели

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset

In [2]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomAffine(0, shear=10, scale=[0.8, 1.2]),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[.485, .456, .406], std=[.229, .224, .225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[.485, .456, .406], std=[.229, .224, .225])
    ])
}


train_dataset = datasets.CIFAR100(root='./SimpleNN', train=True, download=False,
                                  transform=data_transforms['valid'])
aug_dataset = datasets.CIFAR100(root='./SimpleNN', train=True, download=False,
                                transform=data_transforms['train'])
valid_dataset = datasets.CIFAR100(root='./SimpleNN', train=False, download=False,
                                  transform=data_transforms['valid'])


d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
train_dataset

Dataset CIFAR100
    Number of datapoints: 50000
    Root location: ./SimpleNN
    Split: Train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [4]:
aug_dataset

Dataset CIFAR100
    Number of datapoints: 50000
    Root location: ./SimpleNN
    Split: Train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               RandomAffine(degrees=[0.0, 0.0], scale=[0.8, 1.2], shear=[-10.0, 10.0])
               RandomHorizontalFlip(p=0.5)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [5]:
valid_dataset

Dataset CIFAR100
    Number of datapoints: 10000
    Root location: ./SimpleNN
    Split: Test
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [6]:
train_concat = ConcatDataset([train_dataset, aug_dataset])
train_concat

In [7]:
len(train_concat)

100000

In [8]:
train_loader = DataLoader(train_concat, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

In [9]:
model = models.resnet50(pretrained=True)

d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
print(sum(p.numel() for p in model.parameters()))

25557032


In [11]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

'cuda:0'

In [12]:
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(2048, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 100)
)

In [13]:
model = model.to(device)

In [14]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)

In [16]:
def train_model(model, criterion, optimizer, num_epochs=10):
    loss_hist_train = [0] * num_epochs
    accuracy_hist_train = [0] * num_epochs
    loss_hist_valid = [0] * num_epochs
    accuracy_hist_valid = [0] * num_epochs

    for epoch in range(num_epochs):
        model = model.train()
        for idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            predicted_labels = torch.argmax(outputs, 1)
            loss.backward()
            optimizer.step()
            loss_hist_train[epoch] += loss.item() * targets.size(0)
            is_correct = (predicted_labels == targets).sum().item()
            accuracy_hist_train[epoch] += is_correct

            if not idx % 25:
                print('Epoch: %03d|%03d Batch: %03d|%03d Loss: %.4f' % (epoch + 1,
                      num_epochs, idx, len(train_loader), loss.item()))
        loss_hist_train[epoch] /= len(train_loader.dataset)
        accuracy_hist_train[epoch] /= len(train_loader.dataset)

        model = model.eval()
        with torch.inference_mode():
            for idx, (data, targets) in enumerate(valid_loader):
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                loss = criterion(outputs, targets)
                predicted_labels = torch.argmax(outputs, 1)
                loss_hist_valid[epoch] += loss.item() * targets.size(0)
                is_correct = (predicted_labels == targets).sum().item()
                accuracy_hist_valid[epoch] += is_correct

            loss_hist_valid[epoch] /= len(valid_loader.dataset)
            accuracy_hist_valid[epoch] /= len(valid_loader.dataset)
        print(f'Epoch {epoch + 1} | accuracy: ' \
              f'{accuracy_hist_train[epoch]:.4f} | valid accuracy: ' \
              f'{accuracy_hist_valid[epoch]:.4f}')
    return loss_hist_train, loss_hist_valid, accuracy_hist_train, accuracy_hist_valid


In [17]:
history = train_model(model, criterion, optimizer, num_epochs=10)

Epoch: 001|010 Batch: 000|391 Loss: 4.6248
Epoch: 001|010 Batch: 025|391 Loss: 4.2211
Epoch: 001|010 Batch: 050|391 Loss: 3.6325
Epoch: 001|010 Batch: 075|391 Loss: 3.1548
Epoch: 001|010 Batch: 100|391 Loss: 2.8713
Epoch: 001|010 Batch: 125|391 Loss: 2.6720
Epoch: 001|010 Batch: 150|391 Loss: 2.4412
Epoch: 001|010 Batch: 175|391 Loss: 2.2661
Epoch: 001|010 Batch: 200|391 Loss: 2.1947
Epoch: 001|010 Batch: 225|391 Loss: 2.2018
Epoch: 001|010 Batch: 250|391 Loss: 2.0054
Epoch: 001|010 Batch: 275|391 Loss: 2.1635
Epoch: 001|010 Batch: 300|391 Loss: 2.0491
Epoch: 001|010 Batch: 325|391 Loss: 1.9590
Epoch: 001|010 Batch: 350|391 Loss: 1.9167
Epoch: 001|010 Batch: 375|391 Loss: 1.7535
Epoch 1 | accuracy: 0.3657 | valid accuracy: 0.5438
Epoch: 002|010 Batch: 000|391 Loss: 1.9679
Epoch: 002|010 Batch: 025|391 Loss: 1.8726
Epoch: 002|010 Batch: 050|391 Loss: 1.7792
Epoch: 002|010 Batch: 075|391 Loss: 1.9102
Epoch: 002|010 Batch: 100|391 Loss: 1.8921
Epoch: 002|010 Batch: 125|391 Loss: 1.7408
Ep

<h2>Сохранение весов модели

In [18]:
torch.save(model.state_dict(), 'model_weights.pth')
# Есть pt и pth

<h3>Первый подход — просто сохраняем веса

In [20]:
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

<h3>Второй способ — полное сохранение модели

In [21]:
torch.save(model, 'model.pth')

In [23]:
test_model = torch.load('model.pth', weights_only=False)
test_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

<h2>Вектора изображений

In [24]:
vector_model = nn.Sequential(*list(model.children())[:-1])
vector_model

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [30]:
@torch.inference_mode
def inference_data(model, image):
    img = data_transforms['valid'](image).unsqueeze(0)
    model.eval()
    model.to(device)
    img = img.to(device)
    output = model(img)
    output
    return output

In [31]:
from PIL import Image


image = Image.open('D:/MIREA/DPO/Learning/Practic2_7/su57.jpeg')
result = inference_data(vector_model, image)
result

tensor([[[[0.0456]],

         [[1.5535]],

         [[0.8539]],

         ...,

         [[0.3323]],

         [[0.1124]],

         [[0.1477]]]], device='cuda:0')

In [32]:
result.shape

torch.Size([1, 2048, 1, 1])

<h2>Практическое задание</h2>

1. Подобрать любой набор данных на ваш вкус
2. Выполнить обучение с переносом модели VGG19 на выбранных данных

In [3]:
model = models.vgg19(pretrained=True)
model

d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [4]:
print(f'Model: {model.parameters()}')
print(f'Count params: {sum([p.numel() for p in model.parameters()])}')


Model: <generator object Module.parameters at 0x000002545CD802E0>
Count params: 143667240


Ну выберем датасет с цветочками

In [16]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomAffine(0, shear=10, scale=[0.8, 1.2]),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
}

train_dataset = datasets.Flowers102('./Flower102', split='train', download=True,
                                    transform=data_transforms['valid'])
aug_dataset = datasets.Flowers102('./Flower102', split='val', download=True,
                                    transform=data_transforms['train'])
test_dataset = datasets.Flowers102('./Flower102', split='test', download=True,
                                   transform=data_transforms['valid'])

In [6]:
train_dataset

Dataset Flowers102
    Number of datapoints: 1020
    Root location: ./Flower102
    split=train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [7]:
aug_dataset

Dataset Flowers102
    Number of datapoints: 1020
    Root location: ./Flower102
    split=val
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               RandomAffine(degrees=[0.0, 0.0], scale=[0.8, 1.2], shear=[-10.0, 10.0])
               RandomHorizontalFlip(p=0.5)
               RandomVerticalFlip(p=0.3)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [8]:
test_dataset

Dataset Flowers102
    Number of datapoints: 6149
    Root location: ./Flower102
    split=test
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [9]:
train_concat = ConcatDataset([train_dataset, aug_dataset])
len(train_concat)

2040

In [10]:
train_loader = DataLoader(train_concat, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [11]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

'cuda:0'

In [ ]:
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(in_features=25088, out_features=1024),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(1024, 102)
)

In [13]:
model.to(device)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [21]:
from torch.optim.lr_scheduler import StepLR


def train_model_steplr(
    model, 
    criterion, 
    train_loader,
    valid_loader,
    device,
    optimizer=None, 
    num_epochs=10, 
    step_size_lr=5, 
    gamma_lr=0.1
):
    loss_hist_train = [0] * num_epochs
    accuracy_hist_train = [0] * num_epochs
    loss_hist_valid = [0] * num_epochs
    accuracy_hist_valid = [0] * num_epochs
    if (optimizer is None):
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = StepLR(optimizer=optimizer, step_size=step_size_lr, gamma=gamma_lr)

    for epoch in range(num_epochs):
        model = model.train()
        for idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            predicted_labels = torch.argmax(output, 1)
            loss.backward()
            optimizer.step()
            loss_hist_train[epoch] += loss.item() * target.size(0)
            is_correct = (predicted_labels == target).sum().item()
            accuracy_hist_train[epoch] += is_correct

            if not idx % 25:
                print('Epoch: %03d|%03d Batch: %03d|%03d Loss: %.4f' % (epoch + 1,
                      num_epochs, idx, len(train_loader), loss.item()))
        loss_hist_train[epoch] /= len(train_loader.dataset)
        accuracy_hist_train[epoch] /= len(train_loader.dataset)   

        model = model.eval()
        with torch.inference_mode():
            for idx, (data, targets) in enumerate(valid_loader):
                data, targets = data.to(device), targets.to(device)
                outputs = model(data)
                loss = criterion(outputs, targets)
                predicted_labels = torch.argmax(outputs, 1)
                loss_hist_valid[epoch] += loss.item() * targets.size(0)
                is_correct = (predicted_labels == targets).sum().item()
                accuracy_hist_valid[epoch] += is_correct
                if not idx % 50:
                    print('Epoch: %03d|%03d Batch Valid: %03d|%03d Loss Valid: %.4f' % (epoch + 1,
                            num_epochs, idx, len(valid_loader), loss.item()))

            loss_hist_valid[epoch] /= len(valid_loader.dataset)
            accuracy_hist_valid[epoch] /= len(valid_loader.dataset)
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch: {epoch + 1} | accuracy: ' \
              f'{accuracy_hist_train[epoch]:.4f} | valid accuracy: ' \
              f'{accuracy_hist_valid[epoch]:.4f} | lr: {current_lr}')
        scheduler.step()
    return loss_hist_train, loss_hist_valid, accuracy_hist_train, accuracy_hist_valid


In [15]:
criterion = nn.CrossEntropyLoss()
flower_history = train_model_steplr(model, criterion, train_loader,
                                    test_loader, device)

Epoch: 001|010 Batch: 000|064 Loss: 4.5765
Epoch: 001|010 Batch: 025|064 Loss: 3.5519
Epoch: 001|010 Batch: 050|064 Loss: 2.0888
Epoch: 001|010 Batch Valid: 000|097 Loss Valid: 1.1474
Epoch: 001|010 Batch Valid: 050|097 Loss Valid: 1.5230
Epoch 1 | accuracy: 0.2583 | valid accuracy: 0.5783
Epoch: 002|010 Batch: 000|064 Loss: 1.4723
Epoch: 002|010 Batch: 025|064 Loss: 0.5271
Epoch: 002|010 Batch: 050|064 Loss: 1.2613
Epoch: 002|010 Batch Valid: 000|097 Loss Valid: 0.8532
Epoch: 002|010 Batch Valid: 050|097 Loss Valid: 0.5587
Epoch 2 | accuracy: 0.7304 | valid accuracy: 0.6775
Epoch: 003|010 Batch: 000|064 Loss: 0.9225
Epoch: 003|010 Batch: 025|064 Loss: 0.6804
Epoch: 003|010 Batch: 050|064 Loss: 0.9317
Epoch: 003|010 Batch Valid: 000|097 Loss Valid: 0.7304
Epoch: 003|010 Batch Valid: 050|097 Loss Valid: 0.9294
Epoch 3 | accuracy: 0.8250 | valid accuracy: 0.6912
Epoch: 004|010 Batch: 000|064 Loss: 0.4210
Epoch: 004|010 Batch: 025|064 Loss: 0.2976
Epoch: 004|010 Batch: 050|064 Loss: 0.367

In [19]:
model.classifier[2] = nn.Dropout(p=0.5)

In [26]:
train_loader = DataLoader(train_concat, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
flower_history = train_model_steplr(model, criterion, train_loader,
                                    test_loader, device, optimizer=torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-3),
                                    step_size_lr=5, num_epochs=15, gamma_lr=0.5)

Epoch: 001|015 Batch: 000|064 Loss: 0.7025
Epoch: 001|015 Batch: 025|064 Loss: 0.8387
Epoch: 001|015 Batch: 050|064 Loss: 0.0637
Epoch: 001|015 Batch Valid: 000|097 Loss Valid: 0.7741
Epoch: 001|015 Batch Valid: 050|097 Loss Valid: 0.4719
Epoch: 1 | accuracy: 0.8833 | valid accuracy: 0.7673 | lr: 0.0001
Epoch: 002|015 Batch: 000|064 Loss: 0.0203
Epoch: 002|015 Batch: 025|064 Loss: 0.2594
Epoch: 002|015 Batch: 050|064 Loss: 0.2451
Epoch: 002|015 Batch Valid: 000|097 Loss Valid: 0.8635
Epoch: 002|015 Batch Valid: 050|097 Loss Valid: 0.4676
Epoch: 2 | accuracy: 0.9083 | valid accuracy: 0.7759 | lr: 0.0001
Epoch: 003|015 Batch: 000|064 Loss: 0.1552
Epoch: 003|015 Batch: 025|064 Loss: 0.1524
Epoch: 003|015 Batch: 050|064 Loss: 0.2201
Epoch: 003|015 Batch Valid: 000|097 Loss Valid: 0.7736
Epoch: 003|015 Batch Valid: 050|097 Loss Valid: 0.5855
Epoch: 3 | accuracy: 0.9216 | valid accuracy: 0.7792 | lr: 0.0001
Epoch: 004|015 Batch: 000|064 Loss: 0.3006
Epoch: 004|015 Batch: 025|064 Loss: 0.4668

In [ ]:
torch.save(model, 'VGG19_flower102')

In [29]:
import json


with open('flower_history.json', 'w', encoding='utf-8') as f:
    json.dump(flower_history, f, indent=4)

In [ ]:
with open('flower_history.json', 'r', encoding='utf-8') as f:
    loaded_history = json.load(f)